In [1]:
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.types import interrupt
from langgraph.checkpoint.memory import InMemorySaver

from pydantic import BaseModel
from typing import Literal

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

C:\Users\vikas\AppData\Local\Temp\ipykernel_26916\2298160269.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
import sqlite3
from datetime import datetime, date
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
checkpointer = InMemorySaver()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
class IntentClassification(BaseModel):
    intent: Literal["faq", "query", "complaint"]

In [10]:
def get_book(book_id):
    conn = sqlite3.connect("db.sqlite3")
    cursor = conn.cursor()
    
    cursor.execute(
    "SELECT * FROM author_book WHERE id = ?",
    (book_id,)
    )
    
    book = cursor.fetchone()

    if not book:
        raise ValueError(f"Book with id {book_id} not found")

    return book

In [27]:
def get_royalty_earned(book_id: int):
    """Get the royalty earnings for a book."""
    conn = sqlite3.connect("db.sqlite3")
    cursor = conn.cursor()
    
    cursor.execute(
    "SELECT royality_earned FROM author_book WHERE id = ?",
    (book_id,)
    )
    
    royalty = cursor.fetchone()

    if not royalty:
        raise ValueError(f"Book with id {book_id} not found")

    return royalty

In [28]:
get_royality_earned(70)

(28,)

In [29]:
def get_royalty_pending(book_id: int):
    """Get the pending royalty for a book."""

    conn = sqlite3.connect("db.sqlite3")
    cursor = conn.cursor()
    
    cursor.execute(
    "SELECT royality_pending FROM author_book WHERE id = ?",
    (book_id,)
    )
    
    royalty = cursor.fetchone()

    if not royalty:
        raise ValueError(f"Book with id {book_id} not found")

    return royalty

In [30]:
def get_royalty_paid(book_id: int):
    """Get the pending royalty for a book."""

    conn = sqlite3.connect("db.sqlite3")
    cursor = conn.cursor()
    
    cursor.execute(
    "SELECT royality_paid FROM author_book WHERE id = ?",
    (book_id,)
    )
    
    royalty = cursor.fetchone()

    if not royalty:
        raise ValueError(f"Book with id {book_id} not found")

    return royalty

In [31]:
get_royalty_paid(70)

(0,)

In [22]:
get_royality_pending(70)

(2464,)

In [23]:
def get_book_pub_date(book_id: int):
    """Get the current publication status of a book."""

    conn = sqlite3.connect("db.sqlite3")
    cursor = conn.cursor()
    
    cursor.execute(
    "SELECT pub_date FROM author_book WHERE id = ?",
    (book_id,)
    )
    
    pub_date = cursor.fetchone()

    if not pub_date:
        raise ValueError(f"Book with id {book_id} not found")

    return pub_date

In [24]:
get_book_pub_date(70)

('2024-09-10',)

## Tool call

In [32]:
tools = [
    get_royality_earned,
    get_royality_pending,
    get_royalty_paid,
    get_book_pub_date
]

In [33]:
llm_with_tools = llm.bind_tools(tools)

In [ ]:
from langgraph.graph import MessagesState

class State(MessagesState):
    book: int

In [34]:
INTENT_PROMPT = """
You are the intent classifier for BookLeaf Publishing, a publishing company based in New Delhi, India.

Your task is to classify the author's message into exactly ONE of these three intents:

1. FAQ
2. QUERY
3. COMPLAINT

### FAQ

Use FAQ when the author is asking for general information about BookLeaf Publishing, its policies, processes, services, or publishing procedures.

These questions can be answered using the company's knowledge base and do not require accessing the author's specific book or account data.

Examples:

* "What is BookLeaf's royalty policy?"
* "How does the publishing process work?"
* "How long does it take to publish a book?"
* "What are the guidelines for submitting a manuscript?"
* "How does the royalty payment process work?"
* "What formats do you publish books in?"
* "What is your policy for editing?"
* "How can authors submit their manuscripts?"

### QUERY

Use QUERY when the author is asking about information that is specific to their book, account, royalties, publication status, or other data that may require looking up information from a database or using a tool.

Examples:

* "How much royalty have I earned?"
* "How much royalty is pending for my book?"
* "How much royalty has been paid?"
* "Is my book published yet?"
* "When will my book be published?"
* "What is the current status of my book?"
* "How much have I earned from book 101?"
* "Has my royalty been paid?"
* "Is my book live?"

A QUERY generally requires retrieving author-specific or book-specific information rather than answering from general company documentation.

### COMPLAINT

Use COMPLAINT when the author is reporting a problem, error, incorrect information, delay, dissatisfaction, or negative experience with BookLeaf Publishing or its services.

Examples:

* "My royalty payment is missing."
* "My book was supposed to be published but it is still not live."
* "The royalty amount shown is incorrect."
* "I haven't received my royalty payment."
* "My book publication has been delayed."
* "The website is not working."
* "I am unhappy with the publishing service."
* "My book status hasn't been updated."
* "I was promised a payment but haven't received it."

### IMPORTANT DISTINCTIONS

Do not classify based only on keywords. Determine what the author is actually trying to accomplish.

If the author asks about a general BookLeaf policy or process:
→ INFO

If the author asks about their specific book, royalties, account, or publication status:
→ QUERY

If the author reports that something has gone wrong, is incorrect, delayed, missing, or has failed:
→ COMPLAINT

### Borderline examples

"What is the royalty policy?"
→ FAQ

"How much royalty have I earned?"
→ QUERY

"Why haven't I received my royalty?"
→ COMPLAINT

"How does royalty payment work?"
→ FAQ

"When will my royalty be paid?"
→ QUERY

"My royalty payment is overdue."
→ COMPLAINT

"What is the publication process?"
→ FAQ

"When will my book be published?"
→ QUERY

"My book should have been published already."
→ COMPLAINT

"How do I check my book's status?"
→ QUERY

"Can you tell me the status of my book?"
→ QUERY

"My book status hasn't been updated."
→ COMPLAINT

### Output

Return exactly one of:

FAQ
QUERY
COMPLAINT

Do not return explanations, punctuation, or any other text.

Author's message:
{user_input}

"""

In [ ]:
def get_user_intent(state: State) -> Literal["faq", "query", "complaint"]:

    prompt = INTENT_PROMPT.format(user_input= state["user_input"])

    query = state["messages"][-1].content

    prompt += f"\nQuery: {query}"

    structured_llm = llm.with_structured_output(IntentSchema)

    response = structured_llm.invoke(prompt)

    logger.info(
        "Intent classified | book_id=%s | intent=%s | query=%s",
        state["book"],
        response.intent,
        query
    )

    return response.intent

In [ ]:
def assistant(state: State) -> State:

    book_id = state["book"]

    sys_msg = SystemMessage(
        content=(
            "You are a helpful assistant for BookLeaf Publishing. "
            f"You are assisting the author for book ID: {book_id}. "
            "Use the available tools to retrieve book-specific information."
        )
    )

    llm_response = llm_with_tools.invoke(
        [sys_msg] + state["messages"]
    )

    logger.info(
        "Assistant response | book_id=%s | response=%s",
        book_id,
        llm_response.content
    )

    if llm_response.tool_calls:
        logger.info(
            "Tool calls requested | book_id=%s | tool_calls=%s",
            book_id,
            llm_response.tool_calls
        )

    return {
        "messages": [llm_response]
    }

In [ ]:
def register_complaint(state: State) -> State:

    book_id = state["book"]
    query = state["messages"][-1].content

    logger.warning(
        "COMPLAINT | book_id=%s | query=%s",
        book_id,
        query
    )

    response = AIMessage(
        content=(
            "Sorry about that. "
            "Your complaint has been registered and will be reviewed."
        )
    )

    return {
        "messages": [response]
    }

In [ ]:
def get_info(state: State) -> State:

    query = state["messages"][-1].content
    book_id = state["book"]

    logger.info(
        "INFO node | book_id=%s | query=%s",
        book_id,
        query
    )

    search_result = similarity_search(query)

    context = search_result["context"]
    distance = search_result["distance"]

    logger.info(
        "Retrieved knowledge | distance=%s | context=%s",
        distance,
        context
    )

    if distance >= 0.8:

        logger.warning(
            "LOW CONFIDENCE RETRIEVAL | book_id=%s | "
            "query=%s | distance=%s",
            book_id,
            query,
            distance
        )

        # Previously:
        # Ticket.objects.create(...)

        # For now we only log it.
        logger.warning(
            "Human review required | book_id=%s | query=%s",
            book_id,
            query
        )

        response_context = (
            "The knowledge base does not contain enough "
            "information to confidently answer this question."
        )

    else:

        logger.info(
            "Knowledge base result accepted | book_id=%s",
            book_id
        )

        response_context = context

    prompt = INFO_PROMPT

    prompt += (
        f"\nQuery: {query}"
        f"\nResponse: {response_context}"
    )

    llm_response = llm.invoke(prompt)

    logger.info(
        "Info response generated | book_id=%s",
        book_id
    )

    return {
        "messages": [
            AIMessage(content=llm_response.content)
        ]
    }


In [ ]:
def build_graph():

    from langgraph.prebuilt import (
        tools_condition,
        ToolNode,
    )

    builder = StateGraph(State)

    builder.add_node(
        "start_graph",
        start_graph
    )

    builder.add_node(
        "get_info",
        get_info
    )

    builder.add_node(
        "assistant",
        assistant
    )

    builder.add_node(
        "register_complaint",
        register_complaint
    )

    builder.add_node(
        "tools",
        ToolNode(tools)
    )

    # START → start_graph

    builder.add_edge(
        START,
        "start_graph"
    )

    # Intent routing

    builder.add_conditional_edges(
        "start_graph",
        get_user_intent,
        {
            "info": "get_info",
            "query": "assistant",
            "complaint": "register_complaint"
        }
    )

    # Tool routing

    builder.add_conditional_edges(
        "assistant",
        tools_condition,
    )

    builder.add_edge(
        "tools",
        "assistant"
    )

    react_graph = builder.compile(
        checkpointer=checkpointer
    )

    return react_graph

In [ ]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
checkpointer = InMemorySaver()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Define the tools
def get_royality_earned(id):
    """Get the royality earning for a book."""
    book = Book.objects.get(id=id)
    return book.royality_earned

def get_royality_paid(id):
    """Get the royality pending for a book.."""
    book = Book.objects.get(id=id)
    return book.royality_earned

def get_royality_pending(id):
    """Get the royality pending for a book."""
    book = Book.objects.get(id=id)
    return book.royality_pending

def get_book_live_status(id):
    """Get a book current live status
    """
    from datetime import date
    book = Book.objects.get(id=id)
    return f"Already published on {book.pub_date}" if book.pub_date < date.today() else f"Not published yet, publication date: {book.pub_date}"


tools = [get_royality_earned, get_royality_paid,
         get_royality_pending, get_book_live_status]
# Bind the tools
llm_with_tools = llm.bind_tools(tools)

# Define the graph state
class State(MessagesState):
    book: int


def similarity_search(query):
    # get the info chunks
    chunks = documents

    # Create in-memory vector store (FAISS)
    vector_store = FAISS.from_documents(chunks, embeddings)

    # perform similarity search with score
    results = vector_store.similarity_search_with_score(query, k=1)

    # unpack result
    doc, dist = results[0]
    context = doc.page_content

    return {
        "context": context,
        "distance": float(dist)
    }


# Define Nodes
def start_graph(state: State) -> State:
    return state


def get_user_intent(state: State) -> Literal["info", "query", "complaint"]:
    prompt = INTENT_PROMPT
    prompt += f"""/n Query: {state["messages"][0].content}"""
    structured_llm = llm.with_structured_output(IntentSchema)
    response = structured_llm.invoke(prompt)
    return response.intent


def assistant(state: State) -> State:
    # System message
    sys_msg = SystemMessage(
    content=f"You are a helpful assistant tasked with fetching relevant data for the user query. Book id: {state["book"]}")

    llm_response = llm_with_tools.invoke([sys_msg] + state["messages"])
    book = Book.objects.get(id=state["book"])
    Ticket.objects.create(query=state["messages"][0].content,
                          book=book,
                          response=llm_response.content)
    return {"messages": [llm_response]}


def register_complaint(state: State) -> State:
    book, query = state["book"], state["messages"][0].content
    book = Book.objects.get(id=state["book"])
    Ticket.objects.create(query=query,
                          book=book,
                          response='Sorry about that, we have registered your complaint.')
    response = AIMessage(content='Sorry about that, we have registered your complaint.')
    return {"messages": [response]}


def get_info(state: State) -> State:
    query = state["messages"][-1].content
    response = similarity_search(query)
    book = Book.objects.get(id=state["book"])
    if response["distance"] >= 0.8:
        # Save a ticket for Human agent in database
        Ticket.objects.create(query=query,
                              book=book)
    else:
        response = response["context"]
        Ticket.objects.create(query=query,
                              book=book,
                              response=response)

    prompt = INFO_PROMPT

    prompt += f"""/n Query: {query}. Resonse: {response}"""
    llm_response = llm.invoke(prompt)
    return {"messages": [AIMessage(content=llm_response.content)]}


def build_graph():
    from langgraph.prebuilt import tools_condition, ToolNode
    from langgraph.graph import START

    builder = StateGraph(State)

    builder.add_node("start_graph", start_graph)
    builder.add_node("get_info", get_info)

    builder.add_node("assistant", assistant)
    builder.add_node("register_complaint", register_complaint)
    builder.add_node("tools", ToolNode(tools))

    builder.add_edge(START, "start_graph")
    builder.add_conditional_edges(
        "start_graph",
        get_user_intent,
        {
            "info": "get_info",
            "query": "assistant",
            "complaint": "register_complaint"
        }
    )
    builder.add_conditional_edges(
        "assistant",
        # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
        # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
        tools_condition,
    )
    builder.add_edge("tools", "assistant")
    react_graph = builder.compile()

    return react_graph